# ALLaM-7B Najdi Dialect LoRA Fine-Tune — Full Training Notebook

This notebook is the canonical, end-to-end, runnable record of the
ALLaM-7B Najdi dialect LoRA fine-tune for government-services use cases.
It is designed to run top-to-bottom on a **free-tier Google Colab T4 GPU**.

**Plain-English summary:** this notebook downloads a Saudi Arabic model,
teaches it a small "patch" (not a full retrain) so it talks more naturally
in Najdi dialect for government-service questions, and then shows you the
before/after difference. Total cost: about $1 of Colab compute.

Base model: `humain-ai/ALLaM-7B-Instruct-preview`
Dataset: `HeshamHaroon/saudi-dialect-conversations` (filtered to
`government_services` + `customer_service`)


## 1. Install dependencies

**What this does:** installs the exact library versions this project was
built and tested with (see `requirements.txt` in the repo) — `transformers`
for loading the model, `peft` for LoRA adapters, `bitsandbytes` for 4-bit
quantization (QLoRA), `trl` for the supervised fine-tuning trainer, and
`datasets`/`huggingface_hub` for pulling data and the model from the Hub.

**Why it matters:** Colab's preinstalled package versions drift over time
and are frequently incompatible with pinned QLoRA training code. Installing
these exact versions up front avoids version-mismatch errors mid-training
(see `docs/setup.md` for the specific errors this project hit before
pinning these versions).

In [ ]:
!pip install -q transformers==4.44.0 peft==0.12.0 bitsandbytes accelerate==0.33.0 \
    datasets==2.20.0 trl==0.9.6 sentencepiece protobuf huggingface_hub==0.23.2


## 2. Mount Google Drive

**What this does:** mounts your Google Drive inside the Colab VM at
`/content/drive`, so we have a persistent place to write the prepared
dataset (`train.jsonl` / `eval.jsonl`) and the final trained LoRA adapter.

**Why it matters:** Colab VMs are ephemeral — anything written only to the
local VM disk is lost when the runtime disconnects or recycles. Since
training takes tens of minutes and produces an adapter you want to keep,
everything of value in this notebook is saved to Drive, not just local
disk.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# All persistent artifacts (dataset, adapter) live under this folder.
PROJECT_DIR = '/content/drive/MyDrive/allam-najdi-lora'

import os
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f'Project directory: {PROJECT_DIR}')


## 3. Hugging Face authentication

**What this does:** authenticates this Colab session to the Hugging Face
Hub using a token stored in an environment variable, so we can download the
gated `humain-ai/ALLaM-7B-Instruct-preview` model.

**Why it matters:** ALLaM-7B-Instruct-preview requires accepting a license
on its Hugging Face model page before you're allowed to download it (see
`docs/setup.md`, Step 1). We read the token from `os.environ` rather than
hardcoding it in a cell so the notebook can be shared/committed publicly
without leaking credentials — paste your token into the prompt below, it is
not saved into the notebook file.

In [ ]:
import os
from getpass import getpass
from huggingface_hub import login

# Paste your Hugging Face token when prompted (input is hidden).
# Get one at https://huggingface.co/settings/tokens (read access is enough).
os.environ['HF_TOKEN'] = getpass('Enter your Hugging Face token: ')

login(token=os.environ['HF_TOKEN'])
print('Logged in to Hugging Face Hub.')


## 4. Download and prepare the dataset

**What this does:** downloads `HeshamHaroon/saudi-dialect-conversations`
from the Hugging Face Hub, filters it down to only the
`government_services` and `customer_service` conversation categories (the
vertical this project targets), splits the result into train/eval sets, and
saves both as `train.jsonl` / `eval.jsonl` to Google Drive.

**Why it matters:** the source dataset covers many conversation categories,
most of which aren't relevant to a government-services assistant. Filtering
early keeps the fine-tune focused on the target use case instead of
diluting the small training budget across unrelated dialect data. Saving to
Drive (not local Colab disk) means you don't have to re-download and
re-filter if the runtime disconnects.

# ملاحظة: النص العربي في هذه المجموعة موجود في حقل dialect_text
# (نص المستخدم باللهجة النجدية) وحقل response (الرد).

In [ ]:
import json
import random

from datasets import load_dataset

DATASET_NAME = 'HeshamHaroon/saudi-dialect-conversations'
TARGET_CATEGORIES = {'government_services', 'customer_service'}
EVAL_FRACTION = 0.1
RANDOM_SEED = 42

print(f'Downloading dataset: {DATASET_NAME}')
raw = load_dataset(DATASET_NAME, split='train', token=os.environ['HF_TOKEN'])
print(f'Raw dataset size: {len(raw)} examples')

filtered = [ex for ex in raw if ex.get('category') in TARGET_CATEGORIES]
print(f'Filtered to {sorted(TARGET_CATEGORIES)}: {len(filtered)} examples')

normalized = [
    {'prompt': ex['dialect_text'].strip(), 'response': ex['response'].strip()}
    for ex in filtered
]

rng = random.Random(RANDOM_SEED)
rng.shuffle(normalized)

eval_size = max(1, int(len(normalized) * EVAL_FRACTION))
eval_examples = normalized[:eval_size]
train_examples = normalized[eval_size:]

train_path = f'{PROJECT_DIR}/train.jsonl'
eval_path = f'{PROJECT_DIR}/eval.jsonl'

with open(train_path, 'w', encoding='utf-8') as f:
    for ex in train_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + '\n')

with open(eval_path, 'w', encoding='utf-8') as f:
    for ex in eval_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + '\n')

print(f'Wrote {len(train_examples)} train examples to {train_path}')
print(f'Wrote {len(eval_examples)} eval examples to {eval_path}')


## 5. Load ALLaM-7B in 4-bit (QLoRA)

**What this does:** loads `humain-ai/ALLaM-7B-Instruct-preview` quantized
to 4-bit precision (NF4, with double quantization) using `bitsandbytes`,
via a `BitsAndBytesConfig` passed to `from_pretrained`.

**Why it matters:** ALLaM-7B at full 16-bit precision needs roughly 14GB of
VRAM just to hold the weights — too much headroom-wise for a free Colab T4
(15GB total) once you add activations, gradients, and optimizer state.
4-bit quantization shrinks the frozen base model to roughly 4GB, leaving
enough VRAM to train a LoRA adapter on top. See `docs/architecture.md` for
the full memory breakdown.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL_ID = 'humain-ai/ALLaM-7B-Instruct-preview'

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=os.environ['HF_TOKEN'])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=quant_config,
    device_map='auto',
    token=os.environ['HF_TOKEN'],
)

print(f'Loaded {BASE_MODEL_ID} in 4-bit.')
print(f'GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB')


## 6. Attach the LoRA adapter (r=8, alpha=16)

**What this does:** wraps the frozen 4-bit base model with a trainable LoRA
adapter, targeting the attention projection matrices `q_proj`, `k_proj`,
`v_proj`, and `o_proj`.

**Why these settings:** `r=8` is the adapter rank — the largest rank that
comfortably fits a T4's memory budget alongside the 4-bit base model and
still leaves room for a usable batch size. Dialect adaptation is a register
shift, not new-knowledge learning, so a low rank like 8 is expected to be
sufficient (Hu et al., 2021 LoRA paper finds similarly low ranks adequate
for style-type adaptations). `alpha=16` sets the LoRA scaling factor to
`alpha/r = 2`, a standard ratio that gives the adapter a meaningful but not
overwhelming contribution relative to the frozen base. See
`docs/architecture.md` for the full reasoning.

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 7. Train (3 epochs, fp16, T4-optimized)

**What this does:** fine-tunes the LoRA adapter for 3 epochs over the
filtered Najdi government-services training set using `trl`'s
`SFTTrainer`, with settings chosen specifically for a T4's constraints:
`fp16` mixed precision (T4 has no bf16 tensor cores), a small per-device
batch size with gradient accumulation to reach a reasonable effective batch
size, and gradient checkpointing to reduce activation memory.

**Why 3 epochs:** the filtered dataset is small (a government-services /
customer-service subset of one source dataset), so a handful of epochs is
enough to shift the model's register without badly overfitting to specific
phrasings. Training/validation loss per epoch is recorded in
`docs/results.md` — validation loss ticks back up slightly after epoch 1,
which is discussed there as a mild overfitting signal worth watching.

In [ ]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

dataset = load_dataset(
    'json',
    data_files={'train': f'{PROJECT_DIR}/train.jsonl', 'eval': f'{PROJECT_DIR}/eval.jsonl'},
)

def format_example(example):
    return f"### السؤال:\n{example['prompt']}\n\n### الرد:\n{example['response']}"

sft_config = SFTConfig(
    output_dir='/content/adapter_checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    fp16=True,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    report_to='none',
    max_length=512,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset['train'],
    eval_dataset=dataset['eval'],
    formatting_func=format_example,
)

trainer.train()


## 8. Save the adapter to Drive

**What this does:** saves the trained LoRA adapter (and tokenizer) to
Google Drive, not just the ephemeral Colab VM disk.

**Why it matters:** this is the actual deliverable of the whole notebook —
a small adapter directory (~30-80MB, versus ~14GB for the full base model)
that can be downloaded, committed to this repo's release assets, or
uploaded to the Hugging Face Hub. Saving to Drive protects it from being
lost if the Colab runtime disconnects before you've copied it elsewhere.

In [ ]:
ADAPTER_DIR = f'{PROJECT_DIR}/adapter'

trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

import subprocess
size_output = subprocess.run(['du', '-sh', ADAPTER_DIR], capture_output=True, text=True).stdout
print(f'Adapter saved to {ADAPTER_DIR}')
print(f'Adapter size: {size_output.strip()}')


## 9. Evaluate: base vs. fine-tuned side by side

**What this does:** runs the same 3 fixed Najdi government-services
questions through the base model (adapter temporarily disabled) and the
fine-tuned model (adapter enabled), and prints the two responses side by
side for each question.

**Why it matters:** the training/validation loss curves in the previous
cells tell you the model fit the training data, but not whether the output
*actually sounds more Najdi* for the target use case. This qualitative
check is the real test of whether the fine-tune achieved its goal. Full
transcripts and analysis of these same 3 questions are written up in
`docs/results.md`.

In [ ]:
EVAL_QUESTIONS = [
    'وش الأوراق اللي أحتاجها عشان أجدد الإقامة؟',
    'كيف أقدر أحجز موعد في الأحوال المدنية؟',
    'ودي أستفسر عن رسوم تجديد رخصة القيادة، كم تكلفتها؟',
]

def generate(prompt_text, max_new_tokens=200):
    prompt = f"### السؤال:\n{prompt_text}\n\n### الرد:\n"
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )
    full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return full_text[len(prompt):].strip()

for i, question in enumerate(EVAL_QUESTIONS, start=1):
    with model.disable_adapter():
        base_response = generate(question)
    finetuned_response = generate(question)

    print(f'\n[Question {i}] {question}')
    print('-' * 80)
    print(f'BASE ALLaM-7B:\n{base_response}')
    print('-' * 80)
    print(f'FINE-TUNED (Najdi LoRA):\n{finetuned_response}')
    print('=' * 80)
